<a href="https://colab.research.google.com/github/the-name-is-madhu/AI-lab/blob/main/WEEK7_Unfiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import re

def is_variable(expr):
    """Returns True if the expression is a single lowercase letter/variable."""
    return isinstance(expr, str) and expr.islower() and expr.isalpha()

def is_constant(expr):
    """Returns True if the expression is a single uppercase letter/constant."""
    return isinstance(expr, str) and expr.isupper() and expr.isalpha()

def occurs_in(var, expr):
    """Checks if a variable occurs inside another expression (Occurs Check)."""
    if expr == var:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, arg) for arg in expr)
    return False

def apply_subst(subst, expr):
    """Applies a substitution set (dictionary) to an expression."""
    if is_variable(expr):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [apply_subst(subst, arg) for arg in expr]
    return expr

def unify(psi1, psi2):
    # Step 1: If psi1 or psi2 is a variable or constant
    if not isinstance(psi1, list) or not isinstance(psi2, list):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            return {psi2: psi1}
        else:
            return "FAILURE"

    # Step 2: If the initial Predicate/Function symbol is not the same
    if psi1[0] != psi2[0]:
        return "FAILURE"

    # Step 3: If different number of arguments
    if len(psi1) != len(psi2):
        return "FAILURE"

    # Step 4: Set Substitution set (SUBST) to NIL
    subst = {}

    l1 = list(psi1[1:])
    l2 = list(psi2[1:])

    # Step 5: For i=1 to number of elements
    for i in range(len(l1)):
        s = unify(l1[i], l2[i])

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            # Apply S to the remainder of both lists
            l1[i+1:] = [apply_subst(s, item) for item in l1[i+1:]]
            l2[i+1:] = [apply_subst(s, item) for item in l2[i+1:]]

            # Composition: SUBST = APPEND(S, SUBST)
            for key in subst:
                subst[key] = apply_subst(s, subst[key])
            subst.update(s)

    return subst

def parse_expression(text):
    """Parses a string like 'P(x, f(y))' into a nested list structure ['P', 'x', ['f', 'y']]."""
    text = text.replace(" ", "")
    tokens = re.split(r'([(),])', text)
    tokens = [t for t in tokens if t]

    def parse_tokens(index):
        if index >= len(tokens):
            return None, index

        token = tokens[index]
        if token.isalnum():
            # Check if it's a compound expression with arguments
            if index + 1 < len(tokens) and tokens[index + 1] == '(':
                expr = [token]
                index += 2 # skip symbol and '('
                while index < len(tokens) and tokens[index] != ')':
                    arg, index = parse_tokens(index)
                    expr.append(arg)
                    if index < len(tokens) and tokens[index] == ',':
                        index += 1
                return expr, index + 1 # skip ')'
            else:
                return token, index + 1
        return None, index

    result, _ = parse_tokens(0)
    return result

# --- Interactive User Input ---
if __name__ == "__main__":
    print("=== Unification Algorithm ===")
    print("Rule Rules:")
    print(" - Variables: lowercase letters (e.g., x, y, z)")
    print(" - Constants: uppercase letters (e.g., A, B, C)")
    print(" - Format examples: P(x,A)  or  Q(f(x),y)\n")

    try:
        str1 = input("Enter first expression: ")
        str2 = input("Enter second expression: ")

        expr1 = parse_expression(str1)
        expr2 = parse_expression(str2)

        print(f"\nParsed Expression 1: {expr1}")
        print(f"Parsed Expression 2: {expr2}")
        print("-" * 30)

        result = unify(expr1, expr2)

        if result == "FAILURE":
            print("Result: FAILURE (Cannot unify)")
        elif result == {}:
            print("Result: NIL (Already identical, no substitution needed)")
        else:
            print("Result (Substitution Set):")
            for var, val in result.items():
                # Reformat the list back to a string for readable output
                val_str = str(val).replace("[", "").replace("]", "").replace("'", "") if isinstance(val, list) else val
                print(f"  {var} <- {val_str}")

    except Exception as e:
        print(f"Error parsing input. Please ensure syntax matches 'P(x, A)' format.")


=== Unification Algorithm ===
Rule Rules:
 - Variables: lowercase letters (e.g., x, y, z)
 - Constants: uppercase letters (e.g., A, B, C)
 - Format examples: P(x,A)  or  Q(f(x),y)

Enter first expression: P(A,B,B)
Enter second expression: P(x,y,z)

Parsed Expression 1: ['P', 'A', 'B', 'B']
Parsed Expression 2: ['P', 'x', 'y', 'z']
------------------------------
Result (Substitution Set):
  x <- A
  y <- B
  z <- B
